In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
# (50*7)*0.5

#### read the config output per dataset

currently, for each config- we store 1 .pq file per dataset
this notebook concatenates all the outputs into 1 .parquet --> which is the data matrix

this notebook also processes the data matrix for NaN's and filters it, to handle duplicate columns, etc.

In [ ]:
N = 1000
P = 100
dataset_folder = f"syn_data_N{N}_P{P}_10seeds"

# N = 200
# P = 100
# dataset_folder = f'gtex_geneSub_N{N}_P{P}__mode_partition'

# N = 500
# P = 188
# dataset_folder = f'terekhova_N{N}_P{P}_customGenes_oxphos'

# N = 500
# P = 1000
# dataset_folder = f'terekhova_N{N}_P{P}_topK'

# N = 500
# P = 100
# dataset_folder = f'terekhova_N{N}_P{P}_geneSub_K{1000}'

results_dir = Path(os.environ["EPHEMERAL"]) / "nov2026_pyspoc_results/results/"
data_path = os.path.join(results_dir,'per_dataset',dataset_folder)
data_path = os.path.expanduser(data_path)

In [ ]:
results_dir

In [ ]:
configs_lst_all = os.listdir(data_path)

In [ ]:
configs_lst_all

In [ ]:
configs_lst = configs_lst_all

print(len(configs_lst))

##### estimate times to read 1 file

In [ ]:
import pyarrow.dataset as ds

In [ ]:
import time

In [ ]:
# 
configs_file_lens = {}
max_file_len = 0

for config_nm in configs_lst:
    base_dir = os.path.join(data_path,config_nm)
    file_lst_all = [os.path.join(base_dir,filev) for filev in os.listdir(base_dir) if filev.endswith('.parquet')]
    print(len(file_lst_all), ' parquet files found')
    configs_file_lens[config_nm] = len(file_lst_all)
    if len(file_lst_all) > max_file_len:
        max_file_len = len(file_lst_all)       
    # file_lst = [f for f in file_lst_all if os.path.getsize(f)!=0]
    # print(len(file_lst_all), ' non-empty files')    
    t0 = time.perf_counter()
    df = pd.read_parquet(file_lst_all[0])
    print(f"one file: expected {(time.perf_counter() - t0):.3f} s, shape {df.shape}, "
          f"{df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
    print('---')
    # break

#### read the input files in parallel

In [ ]:
from multiprocessing import Pool

def read_pq(f):
    if os.path.getsize(f) == 0:
        return None
    try:
        return pd.read_parquet(f, use_threads=False)
    except Exception:
        return None

In [ ]:
# VERY SLOW! -- POOLED

dfs_dct_config = {}

N_POOLS = 6
N_CHUNK_PER_PID = 20

for config_nm in configs_lst:
    
    print(f'beginning {config_nm}')
    
    dfs_lst = []
    t0 = time.perf_counter()

    base_dir = os.path.join(data_path,config_nm)
    file_lst = [os.path.join(base_dir,filev) for filev in os.listdir(base_dir) if filev.endswith('.parquet')]
    file_lst.sort()
    
    t0 = time.perf_counter()

    with Pool(N_POOLS) as p: 
        dfs_lst = p.map(read_pq, file_lst, chunksize=N_CHUNK_PER_PID)

    
    df = dfs_lst[0]    
    print(f"pool end. {time.perf_counter() - t0:.3f} s, shape {df.shape}, len {len(dfs_lst)}")

    if len(dfs_lst)>0:
        dfs_lst_c = [v for v in dfs_lst if v is not None]
        dfs_dct_config[config_nm] = pd.concat(dfs_lst_c)

    print('---')
    # break

### processing the data matrix

##### drop nan func

In [ ]:
def drop_nans(concat_df, stat_thresh_q, dataset_thresh_q,thresh_quantile=False,print_outs=False):
    """
    input: a concatenated data matrix, with datsets as rows and pyspoc statistics as columns
    `stat_thresh_q`, `dataset_thresh_q` -- thresholds for nan rates
    `thresh_quantile` - false means the thresholds are absolute, true means they are quantiles

    filter the pyspoc data matrix for NaN's in the following way:
        first, column wise:
        - find the fraction of nan values per statistic
        - only keep statistics below `stat_thresh_q` threshold of nan's

        second, row wise:
        - find the fraction of nan values per dataset
        - only keep datasets below `dataset_thresh_q` threshold of nan's
    """
    num_datasets = concat_df.shape[0]
    num_stats = concat_df.shape[1]
    
    old_nas = concat_df.isna().sum().sum()
    
    dct_outs = {}
    dct_outs["old NaNs"] = old_nas
    
    dct_outs["datasets tot"] = num_datasets
    dct_outs["stats tot"] = num_stats

    dct_outs["stats kept"] = num_stats
    dct_outs["datasets kept"] = num_datasets
    
    df = concat_df.copy()
    stats_excluded_df = None
    if old_nas > 0:
        
        stat_frac = df.isna().mean(axis=0)

        if thresh_quantile:
            stat_thresh = stat_frac.quantile(stat_thresh_q)
        else:
            stat_thresh = stat_thresh_q
        
        df = df.loc[:, stat_frac <= stat_thresh]

        ds_frac = df.isna().mean(axis=1)        
        if thresh_quantile:
            dataset_thresh = ds_frac.quantile(dataset_thresh_q)
        else:
            dataset_thresh = dataset_thresh_q

        df = df.loc[ds_frac <= dataset_thresh]

        dct_outs["stats kept"] = df.shape[1]
        dct_outs["datasets kept"] = df.shape[0]
        dct_outs["remaining NaNs"] = df.isna().sum().sum()
        dct_outs["stat_thresh"] = stat_thresh
        dct_outs["dataset_thresh"] = dataset_thresh
        # dct_outs["stats not kept"] = 
        stats_excluded_df = concat_df.loc[:, stat_frac > stat_thresh]

    if print_outs:
        print(dct_outs)
        
    return df,dct_outs, stats_excluded_df

#### filter nan's

In [ ]:
# ONLY KEEP CONFIGS FOR WHICH WE HAVE A MIN NUMBER OF FILES
min_file_num_cnfg = max_file_len*0.75
configs_keep = [v for v in configs_lst if configs_file_lens[v] > min_file_num_cnfg]
num_config = len(configs_keep)

In [ ]:
# CONCAT THE DATA MATRIX FOR ALL KEPT CONFIGS
dfs_concat = {kv : df for kv,df in dfs_dct_config.items() if kv in configs_keep}
all_config_concat = pd.concat(dfs_concat,axis=1,join="outer")
config_all_dict = all_config_concat.columns.groupby(all_config_concat.columns.get_level_values(0))

##### examine nan's across a bunch of stat thresholds

In [ ]:
# DROP NAN'S

stat_kept_lst = []
ds_kept_lst = []
stat_q_lst = []

max_frac_excl = []

for stat_q in np.linspace(0,0.05,11):

    drop_na_concat = drop_nans(all_config_concat,stat_q,data_q)
    
    clean_stats = drop_na_concat[1]

    stat_kept = clean_stats['stats kept']
    ds_kept = clean_stats['datasets kept']

    stat_kept_lst.append(stat_kept)
    ds_kept_lst.append(ds_kept)
    stat_q_lst.append(stat_q)

    excluded_stats = drop_na_concat[2]
    config_excl_dict = excluded_stats.columns.groupby(excluded_stats.columns.get_level_values(0))

    frac_excl_frm_config = [len(ex_s)/len(config_all_dict[kv]) for kv,ex_s in config_excl_dict.items() if ex_s is not None]

    max_frac_excl.append(np.max(frac_excl_frm_config))

In [ ]:
from matplotlib import pyplot as plt

nb_dir = Path("hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/py_scripts/dataviz_notebooks/")
FIG_DIR = Path(os.environ["HOME"]) / nb_dir / 'figures' / dataset_folder
os.makedirs(FIG_DIR,exist_ok=True)

In [ ]:
fig, (ax, axt) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2, 1]})
ax.plot(stat_kept_lst, ds_kept_lst, "o-")
ax.set(xlabel="statistics kept", ylabel="datasets kept")
for i, (s, d) in enumerate(zip(stat_kept_lst, ds_kept_lst)): 
    ax.annotate(str(i), (s, d), xytext=(3, 3), textcoords="offset points", fontsize=8)

axt.axis("off")
tbl = axt.table(cellText=[[i, f"{q:.3f}", s, d, f"{m:.2f}"] for i, (q, s, d, m) in 
                    enumerate(zip(stat_q_lst, stat_kept_lst, ds_kept_lst, max_frac_excl))], 
          colLabels=["#", "stat_q", "stats", "ds.", "max excl"], loc="center",fontsize=10)

tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.3)

figname = f'datamatrix_filtering_table_chosen'
plt.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")

##### choose a threshold and create the filtered df

In [ ]:
stat_q = 0.03
data_q = 0.0

drop_na_concat = drop_nans(all_config_concat,stat_q,data_q)

concat_datamatrix = drop_na_concat[0]
clean_stats = drop_na_concat[1]
excluded_stats = drop_na_concat[2]

In [ ]:
clean_stats

In [ ]:
all_datasets = set.union(*[set(df.index) for kv, df in dfs_dct_config.items()])
excluded_datasets = all_datasets - set(concat_datamatrix.index)
# print(excluded_datasets)
print(len(all_datasets))
print(len(excluded_datasets))
print(set([v.split('__')[0] for v in excluded_datasets]))

In [ ]:
for dv in set([v.split('__')[0] for v in excluded_datasets]):
    print(f'category: {dv}')
    print('num all:',len([v.split('__')[0] for v in all_datasets if dv in v]))
    print('num excluded:',len([v.split('__')[0] for v in excluded_datasets if dv in v]))

note- for synthetic data, the scHPF some of them are very sparse/overdispersed leading to Nans in some stats that don't converge -- especially the graph statistics. currently we exclude the scHPF datasets for which the graph stats fail, just to be able to compare them for other types of data generating processes

In [ ]:
# m.shape

##### save filtered df

In [ ]:
save_concat_df = concat_datamatrix.copy()
if isinstance(save_concat_df.columns, pd.MultiIndex):
    save_concat_df.columns = ["__".join(map(str, c)) for c in save_concat_df.columns]

In [ ]:
import json
from pathlib import Path

print(dataset_folder)
save_folder = os.path.join(os.path.expanduser(results_dir),"concat",dataset_folder)
os.makedirs(save_folder,exist_ok=True)
os.listdir(save_folder)

version=f'filtered_st{int(stat_q*100)}_dt{int(data_q*100)}'

m = save_concat_df

print(save_folder)
m.to_parquet(os.path.join(save_folder, f"datamatrix__{num_config}configs__{version}.parquet"))

record = {
    "configs": clean_stats,
    "shape": list(m.shape),
    "created": pd.Timestamp.now().isoformat(),
    "excluded_datasets_type" : set([v.split('__')[0] for v in excluded_datasets]),
}

with open(os.path.join(save_folder,f"clean_stats__{version}.json"), "w") as f:
    json.dump(record, f, indent=2, default=lambda o : o.item() if hasattr(o, "item") else str(o))

In [ ]:
os.listdir(save_folder)

## FURTHER PROCESSING THE CONCAT DFs

##### remove path files from data matrix column names for external stats

In [ ]:
data_matrix = save_concat_df.copy()

## OPTIONAL: REMOVE PATHS FROM DATA MATRIX COLUMN NAMES
## (EXTERNAL MODULES GET SAVED WITH THE FULL PATH)
stats_path = os.path.expanduser(f'~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/summary_stats/')

data_matrix_names = data_matrix.columns.str.replace(stats_path, ".", regex=False)
data_matrix.columns = data_matrix_names

print(data_matrix.shape[0],data_matrix.shape[1])

In [ ]:
# pd.Series of short col names so they're easier to read
short_stat_names = pd.Series(data_matrix.columns.str.split(".").str[-4:-1].str.join("."), index=data_matrix.columns)

##### remove constant cols

In [ ]:
print('the following cols. are constant across all datasets:')
num_unique_val_per_stat = data_matrix.nunique()
print(short_stat_names[num_unique_val_per_stat <= 1].shape)

# remove constant columns
print('removing constant cols')
df_processed = data_matrix.loc[:,~(num_unique_val_per_stat <= 1)].copy()

#### examining duplicate rows [ie datasets with identical pyspoc vectors]

note- wouldn't expect to see for stochastic data, just as sanity check

In [ ]:
dups_rows = df_processed[df_processed.duplicated(keep=False)] # all columns that have a copy
groups = [[len(list(g.index))] + list(g.index) for _, g in dups_rows.groupby(list(dups_rows.columns), dropna=False)]
groups

In [ ]:
df_processed = df_processed.drop_duplicates().copy()

#### examining duplicate columns [ie duplicate statistic implementations]

In [ ]:
T = df_processed.T
dups = T[T.duplicated(keep=False)] # all columns that have a copy
groups = [[len(list(g.index))] + list(g.index) for _, g in dups.groupby(list(dups.columns), dropna=False)]
groups_short = [[len(list(g.index))] + list(short_stat_names[g.index]) for _, g in dups.groupby(list(dups.columns), dropna=False)]

## ---- FASTER WAY TO FIND DUPS, HASH THE ENTIRE COL
# h = df_processed.apply(lambda c: hash(c.to_numpy().tobytes())) # one hash per column
# groups = [list(cols) for cols in h.groupby(h).groups.values() if len(cols) > 1]
# for g in groups[:10]:
#     print(g)

print(len(groups), "groups of identical columns")
df_dup_cols = pd.DataFrame(groups_short)

In [ ]:
df_dup_cols.sort_values(by=0,ascending=True)
# .to_csv(f'duplicate_cols_{dataset_folder}.csv')

the duplicate cols both measure the same thing: L2 norm of a matrix is its largest singular value


In [ ]:
m.shape,data_matrix.shape,df_processed.shape, df_processed.drop_duplicates().shape

##### remove duplicate stats

In [ ]:
df_dedup = df_processed.T.drop_duplicates().T.copy()

In [ ]:
df_dedup.shape

##### make the dterminant in log scale

In [ ]:
print("NaN:", np.isnan(df_dedup).sum().sum(), " inf:", np.isinf(df_dedup).sum().sum(), " max |x|:", np.nanmax(np.abs(df_dedup)))

In [ ]:
# CURRENTLY I MANUALLY MAKE THE DETERMINNANT LOG
# TO BE IMPLEMENTED IN LIBRARY
detCols = [v for v in df_dedup.columns if 'pyspoc.reducers.basic.Determinant.scaled' in v]

logdet_per_dim = (np.log(df_dedup[detCols].abs().clip(lower=1e-300))) / P
sign = np.sign(df_dedup[detCols])
df_dedup[detCols] = logdet_per_dim
print('log det done!')

In [ ]:
df_processed.shape[1]-len(groups) == df_dedup.shape[1]

In [ ]:
# the intrinsic dimension for some count datasets (very sparse ones) is np.inf
# remove for now
bad = np.isinf(df_dedup.filter(like="id_2nn")).any(axis=1)
print(pd.Series(bad.index[bad].str.split("__").str[0]).value_counts())

if len(bad.index[bad])>0:
    print('flag! infs for intrinsic dim?')
    # df_dedup = df_dedup.loc[~bad,:].copy()

In [ ]:
data_matrix.shape, df_dedup.shape

In [ ]:
(df_dedup.max(axis=0)-df_dedup.min(axis=0)).describe()

In [ ]:
print("NaN:", np.isnan(df_dedup).sum().sum(), " inf:", np.isinf(df_dedup).sum().sum(), " max |x|:", np.nanmax(np.abs(df_dedup)))

In [ ]:
df_dedup.to_parquet(os.path.join(save_folder, f"FILTERED_datamatrix__{len(clean_stats)}configs__{version}.parquet"))

os.listdir(save_folder)

In [ ]:
# # CHECK FOR EXTREMELY BIG VALUES:
# mx = df_dedup.max().sort_values(ascending=False)
# print(mx.head(15))
# big = df_dedup.columns[df_dedup.max() > 1e40]
# print(len(big), "columns above 1e40")
# print(df_dedup[big].idxmax().value_counts().head())

## DIAGNOSTICS CODE [ARCHIVED]

In [ ]:
for kv,ex_s in config_excl_dict.items():
    if ex_s is not None:
        print(kv,'num excl: ',len(ex_s),'frac excl: ', len(ex_s)/len(config_all_dict[kv]))        
        # print(set(ex_s.columns.get_level_values(1)))
        # ex_s_na = ex_s[ex_s.isna()]
        # print(set([v.split('__')[0] for v in ex_s_na.index]))
    else:
        print(kv, 0)

In [ ]:
# EST TIME TO READ 100 FILES
# for config_nm in configs_lst:
#     base_dir = os.path.join(data_path,config_nm)
#     file_lst = [os.path.join(base_dir,filev) for filev in os.listdir(base_dir) if filev.endswith('.parquet')]
#     file_lst.sort()
    
#     t0 = time.perf_counter()

#     with Pool(4) as p: 
#         parts = p.map(read_pq, file_lst[:100], chunksize=10)

#     df = parts[0]
#     print(f"expected {(time.perf_counter() - t0):.3f} s, shape {df.shape}, "
#           f"{df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

In [ ]:
# # VERY SLOW! -- SINGLE THREAD
# dfs_dct_config = {}
# for config_nm in configs_lst:
#     dfs_lst = []
#     t0 = time.perf_counter()
#     for filev in os.listdir(os.path.join(data_path,config_nm)):
#         if filev.endswith('.parquet'):
#             df = pd.read_parquet(os.path.join(data_path,config_nm,filev))
#             dfs_lst.append(df)

#     print(f"{time.perf_counter() - t0:.3f} s, shape {df.shape}, len {len(dfs_lst)}")
    
#     if len(dfs_lst)>0:
#         dfs_dct_config[config_nm] = pd.concat(dfs_lst)

#     break

In [ ]:
# clean = {kv: drop_nans(df,stat_thresh_q=stat_q,dataset_thresh_q=data_q) for kv, df in dfs_dct_config.items()}
# clean = {}

# for kv,df in dfs_dct_config.items():
#     print(kv)
#     clean[kv] = drop_nans(df,stat_thresh_q=stat_q,dataset_thresh_q=data_q) 
    
# clean_dfs = {kv : out_tup[0] for kv,out_tup in clean.items()}
# clean_stats = {kv : out_tup[1] for kv,out_tup in clean.items()}

# clean_dfs_keep = {kv : clean[kv][0] for kv in configs_keep}
# concat_datamatrix = pd.concat(clean_dfs_keep,axis=1,join="inner")

In [ ]:
# FILE READING CODE, USED DS.DATASET WHICH READ FILES IN PARALLEL
# THIS FAILS FOR THE WIDE / MULTI-COLUMN FORMAT
# dfs_dct_config = {}
# for config_nm in configs_lst:
#     base_dir = os.path.join(data_path,config_nm)
#     file_lst = [os.path.join(base_dir,filev) for filev in os.listdir(base_dir) if filev.endswith('.parquet')]
#     dfs_dct_config[config_nm] = ds.dataset(file_lst, format="parquet").to_table().to_pandas()
#     print(config_nm)

In [ ]:
# dfs_dct_config['dist_distrib_stats_metricSweep']

In [ ]:
# PRINTING CODE ---- 
dataset_nan_thresh = 0.9
stat_nan_thresh = 0.9

for kv in ['pyspoc_internal_stats_fast']:
    
    concat_df = dfs_dct_config[kv]
    
    num_datasets = concat_df.shape[0]
    num_stats = concat_df.shape[1]

    print(kv)
    nan_per_dataset = concat_df.isna().sum(axis=1) / num_stats
    
    print('nans per dataset')
    print((nan_per_dataset>0).value_counts())
    print(nan_per_dataset.quantile(dataset_nan_thresh))
    
    nan_per_stat = concat_df.isna().sum(axis=0) / num_datasets

    print('nans per stat')    
    print((nan_per_stat>0).value_counts())
    print(nan_per_stat.quantile(stat_nan_thresh))    

    break

In [ ]:
# os.listdir(os.path.join(data_path,'faster_stats_graph_fractal_cluster'))
# config_nm = 'tk_faster_stats_graph_fractal_cluster'
# fl_nm = 'AS058-1_P13_cells2_geneid0_ctT_N500_P100.parquet'
# oom_pq_fl = os.path.join(data_path,config_nm,fl_nm)
# fine_pq_fl = os.path.join(data_path,'fast_reducedstat_cluster_graph','cpm_counts_gamma_pois__params13__seed1555416210_N1000_P100.parquet')

In [ ]:
# OLD: VERY SLOW!
# dfs_dct_config = {}
# for config_nm in configs_lst:
#     dfs_lst = []
#     for filev in os.listdir(os.path.join(data_path,config_nm)):
#         if filev.endswith('.parquet'):
#             df = pd.read_parquet(os.path.join(data_path,config_nm,filev))
#             dfs_lst.append(df)    
#     if len(dfs_lst)>0:        
#         dfs_dct_config[config_nm] = pd.concat(dfs_lst)

In [ ]:
# fine_pq_fl
'AS062-7_M33_cells1_geneid0_ctT_N500_P1000.parquet'

In [ ]:
oom_pq_fl = f'{results_dir}/per_dataset/terekhova_N500_P1000_topK/dist_distrib_stats_metricSweep/AS062-7_M33_cells1_geneid0_ctT_N500_P1000.parquet'
fine_pq_fl = f'{results_dir}/per_dataset/syn_data_N1000_P100_10seeds/faster_stats_graph_fractal_cluster/cpm_counts_gamma_pois__params13__seed1555416210_N1000_P100.parquet'

In [ ]:
import os, collections
import pyarrow.parquet as pq

t_lst = []
for f in [oom_pq_fl,fine_pq_fl]:
    md = pq.read_metadata(f)
    sch = pq.read_schema(f)
    t=pq.read_table(f).to_pandas()
    t_lst.append(t)
    types = collections.Counter(str(t) for t in sch.types)
    print(os.path.basename(f), "cols:", md.num_columns, "size:", os.path.getsize(f) / 1e6, "MB")
    print("   types:", types.most_common(5))

In [ ]:
t_lst[0]

In [ ]:
t_lst_t = [v.T for v in t_lst]

In [ ]:
t_lst_t[0].shape , t_lst_t[1].shape

In [ ]:
stat = "pyspoc.statistics.ot.wasserstein.PairwiseWassersteinDistance.w1"
print(t_lst[1][stat].columns[-30:-5].tolist()) # what the sub-columns are called
print(t_lst[1][stat].columns.nlevels)

In [ ]:
# ds = np.load('ephemeral/pyspoc_GTEx_data_geneSample/blood/whole_blood/N200_P100__mode_partition/blood__whole_blood__N200_P100_id0_geneid0.npy')
# ds.shape
# (ds==0).sum() / (ds!=0).sum()

In [ ]:
# t_lst_t[0].index.get_level_values(0).value_counts(), t_lst_t[1].index.get_level_values(0).value_counts()

In [ ]:
# t_lst_t[0].drop_duplicates().shape , t_lst_t[1].drop_duplicates().shape

In [ ]:
# t_lst[1]